# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 clinical dataset using the `mlcroissant` library, following the Croissant schema specification. We will reference entities using their `@id` fields for full transparency and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema JSON-LD file at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. The metadata includes essential information and pointers to all available record sets, fields, columns, and distributions, referenced by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subscript or iterate directly)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("Version:", getattr(metadata, 'version', None))
print("Published on:", getattr(metadata, 'datePublished', None))

# List available top-level fields
print("\nAvailable record sets (@id):", getattr(metadata, 'recordSet', []))

## 2. Data Overview
Explore the available record sets, fields, and their `@id`s from the metadata. This step helps identify how to refer to each entity for subsequent data extraction and analysis.

In [ ]:
# List all record sets with their @id
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in metadata.")
else:
    print("Record Sets in dataset:")
    for rs in record_sets:
        print("-", rs)
        # Explore each record set for fields
        rs_obj = dataset.metadata.get(rs)
        if rs_obj is not None:
            fields = getattr(rs_obj, 'field', [])
            print("  Fields (@id):", fields)
        else:
            print("  (Could not retrieve fields for this record set)")

# For demonstration, select first record set to review sample records
if record_sets:
    sample_record_set_id = record_sets[0]
    print(f"\nSample records from record set @id={sample_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load records from each record set into a pandas DataFrame for further exploration. Use the `@id` for the record sets, fields, and columns as obtained above.

In [ ]:
# Extract data from all record sets
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"No records for record set @id={record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns in record set @id={record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

# Choose primary record set for focus (if only one, choose first)
primary_record_set_id = record_sets[0] if record_sets else None
if primary_record_set_id:
    primary_df = dataframes[primary_record_set_id]
else:
    primary_df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. For example, we can select a numeric field using its `@id`, filter records, normalize the field, and group by a categorical variable, also referenced by its `@id`.

**Note:** We use the actual `@id` names as column names, which helps ensure reproducibility and clarity. Replace with the correct `@id`s for numeric and group fields based on your dataset's schema.

In [ ]:
import numpy as np

# Example: Let's assume the numeric field is age and its @id is 'cr:field:age',
# and the group field is anatomical location with @id 'cr:field:anatomical_location'
# In practice, replace these with your actual schema field @id values

numeric_field_id = 'cr:field:age'  # Replace with actual age field @id
group_field_id = 'cr:field:anatomical_location'  # Replace with actual anatomical location field @id

if primary_df is not None:
    # Confirm available columns
    print("\nColumns available:", primary_df.columns.tolist())
    # If the expected fields are present, proceed
    if numeric_field_id in primary_df.columns:
        threshold = 50
        filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if present
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_field_id)
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print(f"Group field {group_field_id} not found in columns.")
    else:
        print(f"Numeric field {numeric_field_id} not found in columns.")

## 5. Visualization
Visualize distributions and relationships between fields in the tabular data. When referencing columns, use their field `@id`.

**Example:** Plot age distribution and age vs anatomical location group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_df is not None and numeric_field_id in primary_df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(primary_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (Age)")
    plt.xlabel("Age")
    plt.ylabel("Frequency")
    plt.show()

    # If group field available, plot age by anatomical group
    if group_field_id in primary_df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=primary_df[group_field_id], y=primary_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id} (Age by Anatomical Location)")
        plt.xlabel("Anatomical Location")
        plt.ylabel("Age")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- Successfully loaded FAIR^2 clinical CRC dataset using Croissant schema and `mlcroissant`.
- Entities referenced by their `@id` ensure reproducible data manipulation.
- Reviewed available record sets and fields; loaded tabular data for EDA and visualization.
- Applied example filtering, normalization, grouping, and plotted key relationships.
- The approach supports transparent clinical research and facilitates FAIR data usage.